# PyTorch framework and internal functions – The torch.nn module

The understanding you have built throughout this crash course will now become even more meaningful as we explore how deep learning is applied in a structured, normalized, and highly efficient manner using PyTorch’s internal modules. Many of the manual steps you previously implemented can be replaced with just a few lines of code using the tools provided in the torch.nn module.

This lecture introduces the most important components of the torch.nn module, explains how they work, and demonstrates how they simplify the construction of neural networks.


## torch.nn.Linear

The nn.Linear layer performs exactly the same operation as the manual expression X @ W + b used in earlier lectures. The difference is that instead of defining W and b as separate tensors, PyTorch packages them into a single, well‑structured layer object.

This layer automatically:
- creates the weight tensor W (2D matrix) and bias vector b,
- stores them as trainable parameters,
- and applies the linear transformation during the forward pass.

In practice, nn.Linear provides a clean, modular, and scalable way to build neural network architectures.


In [68]:
import torch 


# Create a input tensor of shape [10, 1] 
X = torch.randn(10,1)

# ----------
# Create a nn.Linear layer containing W and b:
# ----------
# Input has 1 feature, output has 1 value
D_in = 1
D_out = 1

# Create the linear layer
linear_layer = torch.nn.Linear(in_features=D_in, out_features=D_out)

# Inspect and notice that the paramters W and b is created
# Notice! requires_grad=True is created for us! 
print(f"Layer's Weight (W): {linear_layer.weight}\n")
print(f"Layer's bias (b): {linear_layer.bias}\n")

# Use is just like any python function 
# This is the forward pass
y_hat_nn = linear_layer(X)

print(f"Output of nn.Linear(first 5 rows):\n {y_hat_nn[:3]}")

Layer's Weight (W): Parameter containing:
tensor([[-0.3672]], requires_grad=True)

Layer's bias (b): Parameter containing:
tensor([-0.7551], requires_grad=True)

Output of nn.Linear(first 5 rows):
 tensor([[-0.5402],
        [-0.6494],
        [-0.8219]], grad_fn=<SliceBackward0>)


 ---
## What is a parameter?

A *parameter* is a special kind of tensor used by neural network layers. In PyTorch, a parameter has three important properties:

- **requires_grad=True** by default  
  This means the parameter will be included in gradient computation during backpropagation.

- **Auto‑registers with the model**  
  When you place a parameter inside an nn.Module, PyTorch automatically tracks it as part of the model’s trainable parameters.

- **Participates in the computation graph**  
  PyTorch builds the graph for you, ensuring gradients flow correctly through all operations involving the parameter.

In practice, parameters are the values the model learns during training.


 ---
## Linear models

Linear models are powerful and useful, but they also come with fundamental limitations. Even if you stack ten linear layers on top of each other, the overall transformation is still equivalent to a single linear layer. In other words, linear models can only learn straight‑line relationships.

However, real‑world data is rarely linear. To model complex, noisy, and highly structured patterns, we need to introduce **non‑linearity**. This is achieved by placing a non‑linear **activation function** between our linear layers.

These activation functions allow the network to learn curved, multi‑dimensional, and highly expressive decision boundaries—something linear layers alone can never accomplish.

 ---
## Activation functions

Activation functions introduce the non‑linearity that neural networks need in order to learn complex patterns. Without them, even a deep network would behave like a single linear transformation and would only be able to model straight‑line relationships.

By applying a non‑linear activation between linear layers, the network gains the ability to:

- learn curved and complex relationships,
- build hierarchical feature representations,
- and benefit from stacking multiple layers.

Common activation functions include:

- **ReLU** — simple, fast, and widely used in modern deep networks.
- **GELU** — a smoother, probabilistic version of ReLU, used in modern Transformer models like GPT and Llama.
- **Sigmoid** — compresses values to the range 0–1, often used for binary outputs.
- **Tanh** — outputs values between –1 and 1, useful when centered activations are desired.
- **Softmax** — converts raw scores into probabilities for multi‑class classification.


In summary, activation functions are essential for giving neural networks expressive power. They transform a stack of linear operations into a flexible, non‑linear function approximator capable of modeling real‑world data.

 ---
## ReLU (Non-Linear Rectificed Unit)

In the code below we use the ReLU activation function. ReLU introduces non‑linearity by applying a simple rule: all negative values are replaced with 0, while positive values are kept unchanged. This “breaks” the linear chain created by stacked linear layers and allows the model to learn non‑linear patterns. Without this step, even multiple linear layers would still behave like a single linear transformation.

In [69]:
# The rule is simple: if an input is negative, make it zero
relu = torch.nn.ReLU() 
sample_data = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
activated_data = relu(sample_data) # ReLU(x) = max(0,x)

# See that negative data is now 0
print(f"Original data: {sample_data}")
print(f"Data after ReLu: {activated_data}")


Original data: tensor([-2.0000, -0.5000,  0.0000,  0.5000,  2.0000])
Data after ReLu: tensor([0.0000, 0.0000, 0.0000, 0.5000, 2.0000])


 ---
## GELU (Gaussian Error Linear Unit)

GELU is the modern default activation function used in Transformer architectures such as GPT and Llama. It can be thought of as a smoother, more naturally curved version of ReLU. Instead of abruptly setting all negative values to zero, GELU gradually reduces them based on a Gaussian probability curve.

This smooth transition around zero allows the model to capture subtle patterns and fine‑grained relationships, which is especially important in large language models and other deep architectures.

In the example below we apply the GELU activation function. Unlike ReLU, which sharply sets all negative values to zero, GELU smoothly reduces values around zero based on a Gaussian curve. This creates a gentle, probabilistic transition instead of a hard cutoff. As a result, values near the midpoint are “softened” rather than removed, giving the model a smoother and more expressive non‑linearity.

In [70]:
gelu = torch.nn.GELU()
sample_data = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
activated_data = gelu(sample_data)

# See that all data around the median is smoothed
print(f"Original data: {sample_data}")
print(f"Data after GELU: {activated_data}")

Original data: tensor([-2.0000, -0.5000,  0.0000,  0.5000,  2.0000])
Data after GELU: tensor([-0.0455, -0.1543,  0.0000,  0.3457,  1.9545])


 ---
## Softmax

Softmax works differently from the other activation functions. It is mainly used in the final output layer of a classification model, where it converts the model’s raw scores (**logits**) into a probability distribution. 

This means two things:

1. **All output values will be between 0 and 1**  
2. **All values together will sum to exactly 1**

Because of this, each output can be interpreted as the model’s estimated probability for a given class. For example, an output of 0.72 means “the model is 72% confident in this class.”

Softmax is therefore essential whenever you want the model to choose between multiple classes in a clear and interpretable way.

In [71]:
# We create a Softmax layer. 
# dim=1 means: apply Softmax across each row (each row is one data sample).
softmax = torch.nn.Softmax(dim=1)

# These are raw model outputs called "logits".
# They can be any numbers: negative, positive, large, small.
logits = torch.tensor([
    [1.0, 3.0, 0.5, 1.5],
    [-1.0, 2.0, 1.0, 0.0]
])

# Softmax converts each row of logits into a probability distribution.
# After this step:
# - every value will be between 0 and 1
# - each row will sum to exactly 1
probabilities = softmax(logits)

# Print the resulting probabilities.
# Each number now represents the model's confidence for that class.
print(f"Output probabilities: \n{probabilities}")

# We verify that the probabilities in the first row sum to 1.
# This is what makes Softmax perfect for multi-class classification.
print(f"Sum of probabilities for item 1: {probabilities[0].sum()}")


Output probabilities: 
tensor([[0.0939, 0.6942, 0.0570, 0.1549],
        [0.0321, 0.6439, 0.2369, 0.0871]])
Sum of probabilities for item 1: 1.0


 ---
 ---
## nn.Embedding

nn.Embedding is a PyTorch layer that converts **discrete indices** (such as words, tokens, or category IDs) into **continuous vector representations**. It works like a learned lookup table: each integer maps to a vector that captures semantic information about that token. This mechanism is fundamental in **large language models (LLMs)** such as GPT and Llama, where text must be transformed into numerical form before the model can process it.

Although this layer is *not used* in typical **GeoAI workflows**, such as CNN‑ or U‑Net‑based remote sensing models, it is still valuable to understand. The current AI boom is driven heavily by LLMs, and nn.Embedding is one of the core building blocks behind them. Knowing how it works gives you a broader understanding of modern AI systems and prepares you for work that spans beyond spatial data.

Below is example code demonstrating how nn.Embedding works in practice.

In [72]:
import torch

# We define the size of our "vocabulary".
# In NLP (Natural language processing),
# this would be the number of unique tokens (words, subwords, etc.)
vocab_size = 10

# This is the dimensionality of each embedding vector.
# Each token ID will be mapped to a 3D vector.
embedding_dim = 3

# Create an embedding layer.
# Internally, this is just a matrix of shape [vocab_size, embedding_dim]
# where each row is the vector representation of a token.
embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)

# A "sentence" represented as token IDs.
# In real NLP, these IDs come from a tokenizer.
# Shape: [batch_size, sequence_length]
input_ids = torch.tensor([[1, 5, 0, 8]])

# Passing the IDs through the embedding layer performs a lookup:
# Each ID is replaced with its corresponding embedding vector.
word_vectors = embedding_layer(input_ids)

# Print the input and output to see the transformation.
print(f"Input Token IDs: {input_ids}")
#(bacth_size, sequence_length, embedding_dim)
print(f"Output word vectors: {word_vectors.shape}")
print(f"Output word vectors: {word_vectors}")


Input Token IDs: tensor([[1, 5, 0, 8]])
Output word vectors: torch.Size([1, 4, 3])
Output word vectors: tensor([[[-0.2613,  0.8838,  0.0456],
         [ 0.1256,  0.5707,  0.6913],
         [ 0.6040,  0.0248,  0.5640],
         [-2.8429, -0.6211, -0.5176]]], grad_fn=<EmbeddingBackward0>)


 ---
 ---
## nn.LayerNorm

nn.LayerNorm applies **layer normalization**, a technique that normalizes activations **across the feature dimension** for each individual sample. Unlike BatchNorm, which normalizes across the batch, LayerNorm computes statistics *per sample*, making it independent of batch size. This makes it especially stable when training with very small batches or sequence‑based architectures and prevents values from exploding or vanishing.

LayerNorm is a core component in **transformer‑based models**, including Vision Transformers (ViT), CLIP‑style models, and modern hybrid architectures. Because these models are becoming increasingly important in **spatial AI**, it is useful to understand how LayerNorm works even if you primarily build CNNs or U‑Nets.

Traditional CNN and U‑Net architectures typically rely on **BatchNorm**, but LayerNorm appears more frequently in next‑generation GeoAI models that combine convolutional and transformer‑based components.

Below is example code demonstrating how nn.LayerNorm operates on a tensor.


In [73]:
import torch

# Create a LayerNorm layer.
# normalized_shape=3 means:
# "Normalize over the last dimension, which has length 3."
# In other words, each vector of length 3 will be normalized independently.
norm_layer = torch.nn.LayerNorm(normalized_shape=3)

# Create a small input tensor.
# Shape: (batch=1, sequence_length=2, features=3)
# We have two feature vectors: [1,2,3] and [4,5,6]
# LayerNorm will normalize EACH of these vectors separately.
input_features = torch.tensor([[
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
]])

# Apply LayerNorm.
# For each 3‑element vector, LayerNorm computes:
#   mean = average of the 3 values
#   std  = standard deviation of the 3 values
# Then it normalizes each element:
#   normalized_value = (value - mean) / std
#
# This is done PER VECTOR, not across the batch.
normalized_features = norm_layer(input_features)

# Print the mean of each normalized vector.
# After LayerNorm, each vector should have mean ≈ 0
print(f"Mean: {normalized_features.mean(dim=2)}")

# Print the standard deviation of each normalized vector.
# After LayerNorm, each vector should have std ≈ 1
print(f"Std Dev: {normalized_features.std(dim=2)}")

Mean: tensor([[0., 0.]], grad_fn=<MeanBackward1>)
Std Dev: tensor([[1.2247, 1.2247]], grad_fn=<StdBackward0>)


 ---
 ---
## nn.BatchNorm1d

nn.BatchNorm1d applies **batch normalization**, a technique that normalizes activations **across the batch dimension** for each feature channel. Instead of normalizing each sample independently (as LayerNorm does), BatchNorm computes statistics *across all samples in the batch* for each feature. This makes it highly effective in **CNN‑based architectures**, where feature channels represent learned filters.

BatchNorm has historically been the **default normalization method** in CNNs and U‑Nets, including those widely used in **remote sensing and GeoAI**. It helps stabilize training, accelerates convergence, and reduces internal covariate shift. However, it relies on having a reasonably large batch size—something that can be challenging with very large satellite images.

Even though newer transformer‑based GeoAI models often use LayerNorm instead, BatchNorm remains a core component in many convolutional pipelines and is important foundational knowledge for anyone working with spatial AI.

Below is example code demonstrating how nn.BatchNorm1d operates on a tensor.


In [74]:
import torch

# Create a BatchNorm layer for 3 features.
# BatchNorm1d expects input of shape:
#   (batch_size, num_features, sequence_length)
#
# Here, num_features=3 means:
# "Normalize across the batch for each of the 3 feature channels."
batchnorm_layer = torch.nn.BatchNorm1d(num_features=3)

# Create an input tensor.
# IMPORTANT: BatchNorm expects the feature dimension in the MIDDLE.
#
# Shape: (batch=1, features=3, sequence_length=2)
# This represents 2 feature vectors, each with 3 channels.
input_features = torch.tensor([[
    [1.0, 4.0],   # feature channel 1 across 2 positions
    [2.0, 5.0],   # feature channel 2
    [3.0, 6.0]    # feature channel 3
]])

# Apply BatchNorm.
# BatchNorm computes mean and std ACROSS THE BATCH for each feature channel.
# Since our batch size is 1, BatchNorm will fall back to running estimates.
# (This is why BatchNorm is not ideal for very small batches.)
normalized_features = batchnorm_layer(input_features)

# Print the mean per feature channel.
# We compute mean over the sequence dimension (dim=2).
print(f"Mean per feature channel: {normalized_features.mean(dim=2)}")

# Print the standard deviation per feature channel.
print(f"Std Dev per feature channel: {normalized_features.std(dim=2)}")


Mean per feature channel: tensor([[ 5.9605e-08, -5.9605e-08,  5.9605e-08]], grad_fn=<MeanBackward1>)
Std Dev per feature channel: tensor([[1.4142, 1.4142, 1.4142]], grad_fn=<StdBackward0>)


 ---
 ---
## nn.Dropout

nn.Dropout randomly sets a fraction of the input values to zero **during training**. This prevents the model from relying too heavily on specific activations and acts as a simple form of regularization. The dropout rate determines how much of the input is zeroed out.

When a value is *not* dropped, PyTorch automatically **scales it up** by  

$$
\frac{1}{1 - p}
$$

  
This is called *inverted dropout* and ensures that the **expected output value stays the same** during training and evaluation.  
For example, with p = 0.5, surviving values are multiplied by 2.0.

During evaluation (model.eval()), Dropout is **disabled**, meaning all values pass through unchanged.

Below is example code demonstrating how nn.Dropout behaves during training.

**Note:** This behavior (zeroing + scaling) happens *only* in training mode.


In [75]:
import torch

# Create a Dropout layer with p=0.5
# This means: during training, 50% of the values will be randomly set to zero.
dropout_layer = torch.nn.Dropout(p=0.5)

# Create an input tensor of all ones.
# Shape: (1, 10)
# This makes it easy to see exactly which values Dropout removes.
input_tensor = torch.ones(1, 10)

# -----------------------------
# DROPOUT DURING TRAINING
# -----------------------------
# Enable training mode.
# In this mode, Dropout randomly zeros out values.
dropout_layer.train()

# Apply dropout.
# Roughly half of the values should become zero.
output_during_train = dropout_layer(input_tensor)

print("=== During Training ===")
print(f"Input:  {input_tensor}")
print(f"Output: {output_during_train}\n")

# -----------------------------
# DROPOUT DURING EVAL
# -----------------------------
# Switch to evaluation mode.
# In this mode, Dropout does NOTHING — it passes values through unchanged.
dropout_layer.eval()

output_during_eval = dropout_layer(input_tensor)

print("=== During Evaluation ===")
print(f"Input:  {input_tensor}")
print(f"Output: {output_during_eval}")



=== During Training ===
Input:  tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])
Output: tensor([[0., 2., 2., 0., 2., 0., 2., 2., 2., 2.]])

=== During Evaluation ===
Input:  tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])
Output: tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])


 ---
 ---
## Automating Model Structure and Learning in PyTorch

In the previous lecture, we manually updated weights using .grad and reset gradients with grad.zero_(). This quickly becomes impractical when working with deeper models containing many layers.

PyTorch solves this with two core abstractions:

- **nn.Module** — the structure of the model  
  - Acts as the blueprint that defines layers, parameters, and how data flows through the network.

- **torch.optim** — the learning procedure  
  - Handles weight updates automatically based on the gradients computed during backpropagation.

**Note**: We will now recreate everything we learn with professional PyTorch. No more manual stuff! 

## nn.Module

This pattern is the same for all model development in PyTorch:

- First, we inherit from **torch.nn.Module**  
- Then we define our layers inside the **__init__** method  
- Finally, we connect the layers in the **forward** method  

And that’s it, no multistep secret checklist.  

In [76]:
import torch.nn as nn

# We create our own model class by inheriting from nn.Module.
# This gives us all the machinery PyTorch needs to track parameters,
# register layers, and handle forward passes.
class LinearRegressionModel(nn.Module):

    # The constructor (__init__) runs once when we create the model.
    # Here we define the layers our model will use.
    def __init__(self, in_features, out_features):
        super().__init__()  # Initialize the parent nn.Module class

        # A single linear layer:
        # Takes 'in_features' inputs and produces 'out_features' outputs.
        # For simple linear regression, both are typically 1.
        self.linear_layer = nn.Linear(in_features, out_features)

    # The forward method defines how data flows through the model.
    # PyTorch calls this automatically when we do model(x).
    def forward(self, x):
        # Pass the input x through our linear layer.
        return self.linear_layer(x)

# Create an instance of the model.
# This gives us a fully functional linear regression model.
model = LinearRegressionModel(in_features=1, out_features=1)

# ALl our parameters are now neatly organized! No need for loose tensors
print(f"Model architecture:")
print(model)

Model architecture:
LinearRegressionModel(
  (linear_layer): Linear(in_features=1, out_features=1, bias=True)
)


## torch.optim

Just like nn.Module helps us structure our model, torch.optim helps us automate the learning process.  
Instead of manually updating weights with .grad and resetting gradients with zero_(), PyTorch provides optimizers that handle all of this for us.

The workflow is always the same:

- Create an optimizer and tell it **which parameters to update**
- Compute the loss
- Call loss.backward() to compute gradients
- Let the optimizer update all weights with optimizer.step()
- Reset gradients with optimizer.zero_grad()

One of the most commonly used optimizers is **Adam**, which adapts the learning rate for each parameter and generally works well out of the box.

What Adam actually does under the hood:

- It keeps track of a **running average of the gradients** (first moment)  
  → This acts like momentum and smooths out noisy updates.

- It also keeps a **running average of the squared gradients** (second moment)  
  → This lets Adam estimate how "steep" each parameter direction is.

- Using these two moving averages, Adam computes an **individual learning rate for every parameter**  
  → Parameters with large, unstable gradients get smaller updates.  
  → Parameters with small, stable gradients get larger updates.

- Finally, Adam updates all parameters using these adaptive learning rates, instead of one global learning rate for the whole model.

This makes Adam stable, fast to converge, and effective for many different types of models without much tuning.


Below is example code showing how torch.optim fits into a training loop.

In [77]:
import torch.optim as optim

# We choose a global learning rate.
# This controls how big the weight updates will be.
learning_rate = 0.01

# Create an optimizer.
# Adam is one of the most commonly used optimizers.
# We pass in:
#   - model.parameters(): all weights the optimizer should update
#   - lr=learning_rate: the step size for updates
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Define a loss function.
# MSELoss (Mean Squared Error) is standard for regression tasks.
loss_fn = nn.MSELoss()


 ---
 ---
# The training loop

All these five steps you have learned, and all the messy code, will now be boiled down to the training loop. The engine for the loop is the three following steps

- optimizer.zero_grad()
- loss.backward()
- optimizer.step() 

#### Let's compare the loop we created from scratch with the professional loop

1. Forward pass
    - Scratch
        - y_hat = X @ W +b
    - Professional
        - y_hat = model(x)

2. Loss calculation
    - Scratch
        - loss = torch.mean((y_hat-y_true)**2)
    - Professional
        - loss = loss_fn(y_hat, y_true)

3. Gradient calculation, parameter update, gradient reset 
    - Scratch
        - loss.backward()
        - W -= lr*W.grad
        - b -= lr*b.grad
        - W.grad.zero_()
        - b.grad.zero_()
    - Professional
        - optimizer.zero_grad()
        - loss.backward()
        - optimizer.step()

See the code below and you will see that the professional code gives the same result as our from the scratch model! 

In [ ]:
###################################################
# SYNTETIC DATA FROM LECTURE 6 #
###################################################
# Our batch have 10 samples 
N = 10
# Each sample has 1 input feature and 1 output value 
D_in = 1
D_out = 1

# Create input data
X = torch.randn(N, D_in)
# Create our "ground truth", i.e. the weight and bias
# that will result in a y_hat with the same value as
# our actual y
true_W = torch.tensor([[2.0]]) # true W is 2.0
true_b = torch.tensor(1.0) # true b is 1.0

##!! In real-world applications the model never sees the true W or b.
# We only have input data and the corresponding target labels.
# The model must learn the underlying W and b by comparing its
# predictions to the ground truth and adjusting its parameters
# to minimize the error.

# Create target labels + noise
y_true = X @ true_W + true_b + torch.randn(N, D_out) * 0.1
###########################################################
###########################################################

epochs = 100

for epoch in range(epochs):

    # -------------------------
    # 1. Forward pass
    # -------------------------
    # We send our input x through the model.
    # The model returns predictions y_hat.
    y_hat = model(X)

    # -------------------------
    # 2. Compute loss
    # -------------------------
    # Compare predictions (y_hat) with the true values (y_true).
    # The loss tells us how "wrong" the model is.
    loss = loss_fn(y_hat, y_true)

    # -------------------------
    # 3. Backpropagation engine
    # -------------------------

    # Step 1: Zero the gradients
    # Gradients accumulate by default in PyTorch.
    # We must clear them before computing new ones.
    optimizer.zero_grad()

    # Step 2: Compute gradients
    # This calculates ∂loss/∂weights for every parameter in the model.
    loss.backward()

    # Step 3: Update parameters
    # The optimizer uses the gradients to adjust the weights.
    optimizer.step()

    # -------------------------
    # 4. Print progress
    # -------------------------
    if epoch % 10 == 0:
        # Print epoch number and current loss value.
        # loss.item() extracts the Python float from the tensor.
        print(f"Epoch {epoch:02d}: Loss = {loss.item():.4f}")


Epoch 00: Loss = 3.6856
Epoch 10: Loss = 3.0751
Epoch 20: Loss = 2.5289
Epoch 30: Loss = 2.0512
Epoch 40: Loss = 1.6419
Epoch 50: Loss = 1.2973
Epoch 60: Loss = 1.0118
Epoch 70: Loss = 0.7789
Epoch 80: Loss = 0.5918
Epoch 90: Loss = 0.4437


# Summary: The Concepts Scale — You Don’t Replace Them

Everything we have learned so far transfers directly to larger and more complex models.  
Deep learning in PyTorch is not about adding new conceptual layers — it’s about **scaling up the same building blocks**.

No matter how big the architecture becomes, the core ideas stay the same:

- nn.Module defines the structure of the model  
- forward() defines how data flows through the model  
- loss.backward() computes gradients  
- torch.optim updates the parameters  
- optimizer.zero_grad() resets gradients  
- The training loop repeats these steps  

Whether you build a **1‑layer linear model** or a **100‑layer UNet**, the underlying mechanics do not change.  
You simply stack more layers, add more blocks, and reuse the same principles.

This is why learning PyTorch scales so well:  
Once you understand the fundamentals, you can build anything by **expanding**, not by replacing concepts.

